In [0]:
-- creates new SCD2 table and fills it from source table

CREATE TABLE IF NOT EXISTS first_data_engineering_project.silver.silver_customers_manual_scd2
AS
SELECT
    customer_id,
    city,
    signup_date,
    CURRENT_DATE() AS effective_start_date,
    CAST(NULL AS DATE) AS effective_end_date,
    TRUE AS is_current
FROM first_data_engineering_project.silver.silver_customers;

In [0]:
-- merge finds miss-matched cities and updates the affected rows
MERGE INTO first_data_engineering_project.silver.silver_customers_scd2_manual AS target
USING first_data_engineering_project.silver.silver_customers AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE

WHEN MATCHED AND target.city != source.city THEN
    UPDATE SET effective_end_date = CURRENT_DATE(), is_current = FALSE;

-- creates new row and fills in the new values from source miss-matched data including adding new customers
INSERT INTO first_data_engineering_project.silver.silver_customers_manual_scd2 (
    customer_id, 
    city, 
    signup_date, 
    effective_start_date, 
    effective_end_date, 
    is_current
)

SELECT 
    source.customer_id, 
    source.city, 
    source.signup_date, 
    CURRENT_DATE(),
    NULL,
    TRUE
FROM first_data_engineering_project.silver.silver_customers AS source
LEFT JOIN first_data_engineering_project.silver.silver_customers_manual_scd2 AS target
    ON target.customer_id = source.customer_id
    AND target.is_current = TRUE
    AND target.city = source.city
WHERE target.customer_id IS NULL;